In [5]:
import arcpy

# 1. Define your paths (can be shapefiles, file geodatabase layers, or SDE layers)
source_table = r"C:\Demo\GIS Data\Masaka 2024.gdb\DataProcessing\SewerLines_Updated"
target_table = r"C:\Demo\PostgreSQL-192-sewerassetsdb(NWdataowner).sde\sewerassetsdb.nwdataowner.NWSewer\sewerassetsdb.nwdataowner.SewerLine"

# 2. Define the fields (They must share a common ID match field)
match_field = "GlobalID"        # The common unique ID in both tables
source_value_field = "sewerMaterial"  # The field holding the data you want to copy
target_value_field = "sewerMaterial"  # The empty field you want to calculate/update

# --- Step 1: Read data from the Source Table into a memory dictionary ---
value_lookup = {}

# Use a SearchCursor to read the source data safely
with arcpy.da.SearchCursor(source_table, [match_field, source_value_field]) as src_cursor:
    for row in src_cursor:
        asset_id = row[0]
        value_to_copy = row[1]
        
        # Only add to dictionary if the ID is valid (not null)
        if asset_id is not None:
            value_lookup[asset_id] = value_to_copy

print(f"Successfully cached {len(value_lookup)} records from source table.")

# --- Step 2: Update the Target Table ---
# Use an UpdateCursor to write the values into the target field
with arcpy.da.UpdateCursor(target_table, [match_field, target_value_field]) as tgt_cursor:
    updated_count = 0
    for row in tgt_cursor:
        target_id = row[0]
        
        # Check if this target ID exists in our source lookup dictionary
        if target_id in value_lookup:
            row[1] = value_lookup[target_id]  # Assign the source value to the target field
            tgt_cursor.updateRow(row)          # Commit the change to the database
            updated_count += 1

print(f"Finished. Updated {updated_count} records in the target table.")

Successfully cached 105 records from source table.
Finished. Updated 0 records in the target table.
